## 3.3 MAC 帧类型传输演示

在上一节中，我们学习了三种 MAC 帧类型的结构和 MAC-PHY 适配原理。本节先拆解数据帧从 MAC 字节到 IQ 信号的完整 PHY 流水线，再演示信令帧和复用帧。

### 本实验涉及的关键文件

```
src/nearlink_sdr/
├── mac/
│   ├── frame.py             <- AsyncDataFrame: 异步数据帧 / MuxFrame: 复用帧
│   └── signaling.py         <- encode_signaling: 控制信令编码
├── phy/
│   ├── mac_interface.py     <- mac_to_iq: MAC字节->IQ / iq_to_mac: IQ->MAC字节
│   │                           signaling_to_iq / iq_to_signaling: 信令帧转换
│   ├── tx_pipeline.py       <- TxConfig: 发射配置 / encode_head/encode_payload: 编码
│   │                           _pulse_shape_symbols: QPSK调制+RRC
│   ├── frame.py             <- assemble_frame_bits: 帧组装 / frame_to_symbols: 比特->符号
│   │                           symbols_to_data_bits: 符号->比特
│   └── rx_pipeline.py       <- decode_head / decode_payload: Polar解码+CRC校验
```


---

### 1. 导入与参数

In [ ]:
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame, MuxFrame
from nearlink_sdr.mac.link_control import PingRequest
from nearlink_sdr.mac.signaling import encode_signaling
from nearlink_sdr.phy.tx_pipeline import TxConfig, encode_head, encode_payload, _pulse_shape_symbols, _HEAD_CODED_LEN
from nearlink_sdr.phy.mac_interface import bytes_to_bits, bits_to_bytes, mac_to_iq
from nearlink_sdr.phy.control_info import ControlInfoA2
from nearlink_sdr.phy.channel import ChannelModel
from nearlink_sdr.phy.frame import assemble_frame_bits, frame_to_symbols, symbols_to_data_bits, FrameConfig
from nearlink_sdr.phy.psk import PSKModulator, rrc_filter
from nearlink_sdr.phy.rx_pipeline import decode_head, decode_payload
from nearlink_sdr.common.mcs import get_mcs
from nearlink_sdr.common.code_block_seg import segment_without_crc
from nearlink_sdr.phy.mac_interface import iq_to_mac, iq_to_signaling, signaling_to_iq

cfg = TxConfig(
    frame_type=2, mcs_index=7, pid=0x123456,
    whitening_seed=0x52, crc_seed=0x555555,
    crc_len=24, ctrl_bits_len=28, pilot_interval=8,
)
ch = ChannelModel(snr_db=10.0)
rng = np.random.default_rng(42)
payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))

---

### 2. 数据帧：发送端 (mac_to_iq)

以下展示 mac_to_iq 的具体过程：

In [ ]:
# 步骤1: 组帧
data_frame = AsyncDataFrame(segment_type=0, data=payload)
mac_bytes = data_frame.pack()
print(f"MAC 帧: {len(mac_bytes)} bytes")

# 步骤2: bytes -> bits + 控制信息
data_bits = bytes_to_bits(mac_bytes)
ctrl_info = ControlInfoA2(packet_type=0, empty_packet=0, tx_sn=0, rx_sn=0, flow_ctrl=0, sys_mgmt_rx=0, reserved=0, data_length=len(mac_bytes))
ctrl_bits = ctrl_info.pack(sync_seed=cfg.pid)
print(f"数据比特: {len(data_bits)}, 控制信息比特: {len(ctrl_bits)}")

# 步骤3: Polar 编码 (头+载荷分别编码)
head_w = encode_head(ctrl_bits, cfg)
payload_w = encode_payload(data_bits, cfg)
print(f"编码后: head={len(head_w)} bits, payload={len(payload_w)} bits")

# 步骤4: 帧组装 (preamble+sync+head+pilots+payload)
frame_cfg = FrameConfig(
    frame_type=2, symbol_rate_mhz=1.0, pilot_interval=8,
    crc_len=24, mod_type=cfg.mod_str,
)
fields = assemble_frame_bits(head_w, payload_w, frame_cfg)
symbols = frame_to_symbols(fields, frame_cfg)
print(f"帧符号数: {len(symbols)}")

# 步骤5: QPSK调制 + RRC -> IQ
mod = PSKModulator(mod_type=cfg.mod_str, sps=cfg.sps)
iq = _pulse_shape_symbols(symbols, mod)
print(f"IQ: {len(iq)} samples")

---

### 3. 数据帧：接收端 (iq_to_mac)

In [ ]:
# 步骤6: AWGN 信道
rx_iq = ch.apply_awgn(iq, cfg.sps)
print(f"AWGN: SNR=10 dB")

以下展示 iq_to_mac 的具体过程：

In [4]:
# 步骤7: 匹配滤波 + 下采样
h_rrc = rrc_filter(0.4, cfg.sps)
filtered = np.convolve(rx_iq, h_rrc, mode="same")
symbols_rx = filtered[::cfg.sps]

In [ ]:
# 步骤8: 帧解析
# 利用 preamble/sync 定位帧边界, 提取控制头和数据部分的编码比特
head_coded = _HEAD_CODED_LEN.get(2, cfg.ctrl_bits_len + 12)
mcs_e = get_mcs(cfg.mcs_index)
b_len = len(data_bits) + cfg.crc_len
dummy = np.zeros(b_len, dtype=int)
segments = segment_without_crc(dummy, str(mcs_e.code_rate), crc_len=cfg.crc_len)
total_coded = sum(n for n, _ in segments)

frame_cfg2 = FrameConfig(
    frame_type=2, symbol_rate_mhz=1.0, pilot_interval=8,
    crc_len=24, mod_type=cfg.mod_str,
)
ctrl_bits_rx, data_bits_rx = symbols_to_data_bits(
    symbols_rx, frame_cfg2, n_ctrl_coded_bits=head_coded, n_data_bits=total_coded,
)
print(f"帧解析: ctrl={len(ctrl_bits_rx)} bits, data={len(data_bits_rx)} bits")

In [ ]:
# 步骤9: 头部解码 + 载荷解码
# Polar解码 + CRC校验
ctrl_decoded, head_ok = decode_head(ctrl_bits_rx, cfg)
data_decoded, crc_ok = decode_payload(data_bits_rx, cfg, len(data_bits))
print(f"Head CRC: {'OK' if head_ok else 'FAIL'}")
print(f"Data CRC: {'OK' if crc_ok else 'FAIL'}")

# 步骤10: bits -> MAC bytes
mac_recovered = bits_to_bytes(data_decoded[:len(mac_bytes) * 8])
print(f"Payload match: {mac_recovered == mac_bytes}")

---

### 4. 信令帧

信令帧不同于数据帧——它承载的是 MAC 层控制信令（如 Ping、接入请求/响应），而非用户数据。转换使用专用函数 `signaling_to_iq` / `iq_to_signaling`：

**`signaling_to_iq(msg, cfg)`** — 信令对象 → ControlFrame → `mac_to_iq` → IQ 信号。内部三步：`encode_signaling(msg)` 将信令对象编码为控制帧 → `frame.pack()` 序列化为字节 → `mac_to_iq()` 将字节转为 IQ 信号。

```python
def signaling_to_iq(msg: object, cfg: TxConfig) -> np.ndarray:
    frame = encode_signaling(msg)          # 信令对象 -> ControlFrame
    mac_bytes = frame.pack()               # ControlFrame -> 字节序列
    return mac_to_iq(mac_bytes, cfg)       # 字节 -> PHY 管线 -> IQ
```

其中 `encode_signaling(msg)`（位于 `signaling.py`）根据信令类型查找对应的 Data Type Index，调用对应的 `ControlFrame` 编码方法，返回包含帧头和载荷的控制帧对象。`mac_to_iq()` 是数据帧和信令帧共用的 MAC→PHY 管线（见上方 2-3 节的 10 步拆解）。

---

**`iq_to_signaling(iq_signal, cfg, n_mac_bytes)`** — IQ 信号 → `iq_to_mac` → `ControlFrame.unpack` → `decode_signaling` → 信令对象。

```python
def iq_to_signaling(iq_signal: np.ndarray, cfg: TxConfig,
                    n_mac_bytes: int) -> tuple[object, bool]:
    rx = iq_to_mac(iq_signal, cfg, n_mac_bytes)   # IQ -> MacRxResult
    if not rx.crc_ok:
        return None, False                         # CRC 失败 -> 丢弃
    frame, _ = ControlFrame.unpack(rx.mac_payload)  # 字节 -> ControlFrame
    msg = decode_signaling(frame)                   # ControlFrame -> 信令对象
    return msg, True
```

其中 `ControlFrame.unpack(mac_payload)` 将字节流解析为控制帧结构（提取 Data Type Index、载荷字段），`decode_signaling(frame)` 根据 Data Type Index 查表反向解码为具体的信令 Python 对象（如 PingRequest、PowerControlRequest 等）。

---

下面是实际调用演示：

In [ ]:
msg = PingRequest()
iq = signaling_to_iq(msg, cfg)
rx_iq = ch.apply_awgn(iq)
recovered, ok = iq_to_signaling(rx_iq, cfg, len(encode_signaling(msg).pack()))
print(f"[Signaling] CRC={'OK' if ok else 'FAIL'}")

---

### 5. 复用帧

复用帧（`MuxFrame`）是信令帧和数据帧的容器——将多个控制帧和一个可选数据帧拼成一条 MAC 字节流后统一走 `mac_to_iq` 发送。

```python
class MuxFrame:
    control_frames: list[ControlFrame]               # 控制面帧列表 (可多个)
    data_frame: AsyncDataFrame | SyncDataFrame | None = None  # 数据面帧 (可选)

    def pack(self) -> bytes:
        result = bytearray()
        for cf in self.control_frames:
            result.extend(cf.pack())                 # 逐个控制帧序列化
        if self.data_frame is not None:
            result.extend(self.data_frame.pack())    # 数据帧追加在末尾
        return bytes(result)
```

接收端 `iq_to_mac` 还原出完整的 `mac_payload` 后，按已知的控制帧长度从头部切出各控制帧，剩余部分即为数据帧载荷。复用帧本身不引入新的发射管线——`mac_to_iq` / `iq_to_mac` 与数据帧完全相同，只是 MAC 字节的组成方式不同。

In [ ]:
ctrl_frame = encode_signaling(PingRequest())
mux = MuxFrame(control_frames=[ctrl_frame], data_frame=data_frame)
mac_bytes = mux.pack()
iq = mac_to_iq(mac_bytes, cfg)
rx_iq = ch.apply_awgn(iq, cfg.sps)
rx = iq_to_mac(rx_iq, cfg, len(mac_bytes))
ctrl_len = len(ctrl_frame.pack())
data_ok = rx.mac_payload[ctrl_len:] == payload if rx.crc_ok else False
print(f"[Mux] CRC={'OK' if rx.crc_ok else 'FAIL'}, data match={data_ok}")

---

### 课后代码实践

请补全下方 mac_to_iq 发射管线中的 **4 处空缺**（每处一行代码），完成从载荷到 IQ 信号的完整调制流程。

要求：

1. 补全数据帧的创建与序列化（2 处）
2. 补全载荷 Polar 编码（1 处）
3. 补全帧字段到 QPSK 调制符号的映射（1 处）

完成后运行  验证 IQ 信号是否生成成功。

In [ ]:
%%writefile mac_pipeline_practice.py
import sys
sys.path.insert(0, "../src")
import numpy as np
from nearlink_sdr.mac.frame import AsyncDataFrame
from nearlink_sdr.phy.tx_pipeline import TxConfig, encode_head, encode_payload, _pulse_shape_symbols
from nearlink_sdr.phy.mac_interface import bytes_to_bits
from nearlink_sdr.phy.control_info import ControlInfoA2
from nearlink_sdr.phy.frame import assemble_frame_bits, frame_to_symbols, FrameConfig
from nearlink_sdr.phy.psk import PSKModulator

cfg = TxConfig(frame_type=2, mcs_index=7, pid=0x123456,
               whitening_seed=0x52, crc_seed=0x555555,
               crc_len=24, ctrl_bits_len=28, pilot_interval=8)
rng = np.random.default_rng(42)
payload = bytes(rng.integers(0, 256, 10, dtype=np.uint8))

# ==== TODO: 补全 mac_to_iq 管线的 4 处空缺 ====

# 步骤 1: 创建数据帧 + 序列化为字节
frame =                                   # 1: 创建 AsyncDataFrame （补全）
mac_bytes =                                    #  2: 序列化为 MAC 字节 （补全）

# 步骤 2: 字节 -> 比特 + 控制信息
data_bits = bytes_to_bits(mac_bytes)
ctrl_info = ControlInfoA2(packet_type=0, empty_packet=0,
    tx_sn=0, rx_sn=0, flow_ctrl=0, sys_mgmt_rx=0,
    reserved=0, data_length=len(mac_bytes))
ctrl_bits = ctrl_info.pack(sync_seed=cfg.pid)
head_w = encode_head(ctrl_bits, cfg)

# 步骤 3: 载荷 Polar 编码
payload_w =                                    #  3: 载荷比特编码 （补全）

# 步骤 4: 帧组装 -> 符号映射 -> IQ
frame_cfg = FrameConfig(frame_type=2, symbol_rate_mhz=1.0,
    pilot_interval=8, crc_len=24, mod_type=cfg.mod_str)
fields = assemble_frame_bits(head_w, payload_w, frame_cfg)
symbols =                                    #  4: 比特 -> QPSK 符号 （补全）
mod = PSKModulator(mod_type=cfg.mod_str, sps=cfg.sps)
iq = _pulse_shape_symbols(symbols, mod)

print(f"IQ signal: {len(iq)} samples")
print("PASS" if len(iq) > 0 else "FAIL")


执行以下命令进行编译并验证结果：


In [ ]:
!python mac_pipeline_practice.py


In [ ]:
!cat answer/03.03_answer.txt